In [7]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score, accuracy_score
from PIL import Image
import timm

# --- CONFIG ---
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
BASE_PATH = '/kaggle/input/datasets/mariaherrerot/aptos2019/'
TRAIN_IMG_DIR = os.path.join(BASE_PATH, 'train_images/train_images')
VAL_IMG_DIR = os.path.join(BASE_PATH, 'val_images/val_images')
IMG_SIZE = 256
BATCH_SIZE = 24 # Reduced for deeper gradients
LR = 5e-5

# --- NOVELTY: CONTRASTIVE ATTENTION HEAD ---
class ContrastiveAptosNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Backbone: ConvNeXt-Tiny (Superior for fine-grained medical textures)
        self.backbone = timm.create_model('convnext_tiny', pretrained=True, num_classes=0)
        dim = 768 # ConvNeXt Tiny output dim
        
        # Attention Gate
        self.attn = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Sigmoid()
        )
        
        # Projector for Contrastive separation (The Novelty)
        self.projector = nn.Sequential(
            nn.Linear(dim, 512),
            nn.ReLU(),
            nn.Linear(512, 128)
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(dim),
            nn.Dropout(0.5),
            nn.Linear(dim, 5)
        )

    def forward(self, x):
        feat = self.backbone(x)
        attn_weight = self.attn(feat)
        refined_feat = feat * attn_weight
        
        logits = self.classifier(refined_feat)
        proj = self.projector(refined_feat)
        return logits, proj

# --- CUSTOM LOSS: CrossEntropy + Contrastive Regularization ---
class AptosLoss(nn.Module):
    def __init__(self, alpha=0.1):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(label_smoothing=0.1)
        self.alpha = alpha

    def forward(self, logits, projs, labels):
        ce_loss = self.ce(logits, labels)
        # Simplified Contrastive: Penalize small variance in projections
        reg_loss = 1.0 / (torch.var(projs) + 1e-6)
        return ce_loss + self.alpha * reg_loss

# --- ENGINE ---
def train_novel_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ContrastiveAptosNet().to(device)

    # RE-ADJUSTED SAMPLER: Focus exclusively on Severe (Class 3)
    train_df = pd.read_csv(os.path.join(BASE_PATH, 'train_1.csv'))
    counts = train_df.diagnosis.value_counts()
    class_weights = 1. / counts
    class_weights[3] *= 4.0 # Aggressive upweighting for Stage 3
    sample_weights = class_weights[train_df.diagnosis].values
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(0.2, 0.2, 0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    train_loader = DataLoader(AptosDataset('train_1.csv', TRAIN_IMG_DIR, train_tf), 
                              batch_size=BATCH_SIZE, sampler=sampler, num_workers=4)
    val_loader = DataLoader(AptosDataset('valid.csv', VAL_IMG_DIR, train_tf), batch_size=BATCH_SIZE)

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
    criterion = AptosLoss(alpha=0.05)
    
    best_kappa = 0
    for epoch in range(15):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            logits, projs = model(imgs)
            loss = criterion(logits, projs, labels)
            loss.backward()
            optimizer.step()

        model.eval()
        v_preds, v_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                logits, _ = model(imgs)
                v_preds.extend(torch.argmax(logits, 1).cpu().numpy())
                v_labels.extend(labels.cpu().numpy())
        
        kappa = cohen_kappa_score(v_labels, v_preds, weights='quadratic')
        print(f"Epoch {epoch+1} | Kappa: {kappa:.4f} | Acc: {accuracy_score(v_labels, v_preds):.4f}")
        
        if kappa > best_kappa:
            best_kappa = kappa
            torch.save(model.state_dict(), 'novel_contrastive_aptos.pth')

    # FINAL DOCS
    print("\n" + "="*50)
    print("      RESEARCH-READY PERFORMANCE REPORT")
    print("="*50)
    model.load_state_dict(torch.load('novel_contrastive_aptos.pth'))
    model.eval()
    
    f_preds, f_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits, _ = model(imgs)
            f_preds.extend(torch.argmax(logits, 1).cpu().numpy())
            f_labels.extend(labels.cpu().numpy())

    cm = confusion_matrix(f_labels, f_preds)
    classes = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
    
    for i in range(5):
        tp = cm[i, i]
        fn = sum(cm[i, :]) - tp
        fp = sum(cm[:, i]) - tp
        tn = sum(cm.flatten()) - (tp + fn + fp)
        print(f"{classes[i]:<15} | Sens: {tp/(tp+fn):.4f} | Spec: {tn/(tn+fp):.4f}")
    
    print("-" * 50)
    print(f"Final Quadratic Kappa: {cohen_kappa_score(f_labels, f_preds, weights='quadratic'):.4f}")

# Re-use the AptosDataset class from previous runs
if __name__ == "__main__":
    train_novel_model()

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Epoch 1 | Kappa: 0.7916 | Acc: 0.6475
Epoch 2 | Kappa: 0.8822 | Acc: 0.7568
Epoch 3 | Kappa: 0.7883 | Acc: 0.7049
Epoch 4 | Kappa: 0.8375 | Acc: 0.7514
Epoch 5 | Kappa: 0.8527 | Acc: 0.7842
Epoch 6 | Kappa: 0.8860 | Acc: 0.8169
Epoch 7 | Kappa: 0.8529 | Acc: 0.7568
Epoch 8 | Kappa: 0.8517 | Acc: 0.7760
Epoch 9 | Kappa: 0.7966 | Acc: 0.7240
Epoch 10 | Kappa: 0.8417 | Acc: 0.7104
Epoch 11 | Kappa: 0.8449 | Acc: 0.6995
Epoch 12 | Kappa: 0.8638 | Acc: 0.7295
Epoch 13 | Kappa: 0.8086 | Acc: 0.7049
Epoch 14 | Kappa: 0.8277 | Acc: 0.7486
Epoch 15 | Kappa: 0.8612 | Acc: 0.8142

      RESEARCH-READY PERFORMANCE REPORT
No DR           | Sens: 0.9942 | Spec: 0.9948
Mild            | Sens: 0.5750 | Spec: 0.9632
Moderate        | Sens: 0.7885 | Spec: 0.9008
Severe          | Sens: 0.7273 | Spec: 0.9390
Proliferative   | Sens: 0.3571 | Spec: 0.9882
--------------------------------------------------
Final Quadratic Kappa: 0.8942


In [8]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score, accuracy_score, f1_score
from PIL import Image
import timm

# --- SETTINGS ---
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
BASE_PATH = '/kaggle/input/datasets/mariaherrerot/aptos2019/'
VAL_IMG_DIR = os.path.join(BASE_PATH, 'val_images/val_images')
MODEL_PATH = 'novel_contrastive_aptos.pth'
IMG_SIZE = 256
BATCH_SIZE = 32

# --- ARCHITECTURE RECOVERY ---
class ContrastiveAptosNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('convnext_tiny', pretrained=False, num_classes=0)
        dim = 768 
        self.attn = nn.Sequential(nn.Linear(dim, dim), nn.Sigmoid())
        self.projector = nn.Sequential(nn.Linear(dim, 512), nn.ReLU(), nn.Linear(512, 128))
        self.classifier = nn.Sequential(nn.BatchNorm1d(dim), nn.Dropout(0.5), nn.Linear(dim, 5))

    def forward(self, x):
        feat = self.backbone(x)
        attn_weight = self.attn(feat)
        refined_feat = feat * attn_weight
        return self.classifier(refined_feat), self.projector(refined_feat)

class AptosEvalDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(os.path.join(BASE_PATH, csv_file))
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, f"{self.df.iloc[idx, 0]}.png")
        image = Image.open(img_path).convert('RGB')
        label = self.df.iloc[idx, 1]
        if self.transform: image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

# --- EXECUTION ---
def final_evaluation():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ContrastiveAptosNet().to(device)
    
    if os.path.exists(MODEL_PATH):
        model.load_state_dict(torch.load(MODEL_PATH))
        model.eval()
        print(f"Model successfully loaded from {MODEL_PATH}")
    else:
        print("Model file not found. Ensure training finished successfully.")
        return

    test_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    test_loader = DataLoader(AptosEvalDataset('valid.csv', VAL_IMG_DIR, test_tf), batch_size=BATCH_SIZE)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(device)
            logits, _ = model(imgs)
            all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
            all_labels.extend(labels.numpy())

    # --- CALCULATE FINAL METRICS ---
    acc = accuracy_score(all_labels, all_preds)
    kappa = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    cm = confusion_matrix(all_labels, all_preds)
    classes = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

    print("\n" + "!"*60)
    print("             FINAL DISSERTATION METRICS REPORT")
    print("!"*60)
    print(f"FINAL TESTING ACCURACY:  {acc:.4%}")
    print(f"FINAL QUADRATIC KAPPA:   {kappa:.4f}")
    print("-" * 60)

    stats = []
    for i in range(5):
        tp = cm[i, i]
        fn = sum(cm[i, :]) - tp
        fp = sum(cm[:, i]) - tp
        tn = sum(cm.flatten()) - (tp + fn + fp)
        
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        f1 = f1_score(all_labels, all_preds, average=None)[i]
        
        stats.append([classes[i], sens, spec, f1])

    report_df = pd.DataFrame(stats, columns=['Class', 'Sensitivity', 'Specificity', 'F1-Score'])
    print(report_df.to_string(index=False))
    
    print("\nFull Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=classes))
    print("!"*60)

if __name__ == "__main__":
    final_evaluation()

Model successfully loaded from novel_contrastive_aptos.pth

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
             FINAL DISSERTATION METRICS REPORT
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
FINAL TESTING ACCURACY:  82.5137%
FINAL QUADRATIC KAPPA:   0.8881
------------------------------------------------------------
        Class  Sensitivity  Specificity  F1-Score
        No DR     0.982558     1.000000  0.991202
         Mild     0.650000     0.960123  0.658228
     Moderate     0.778846     0.908397  0.775120
       Severe     0.772727     0.930233  0.539683
Proliferative     0.321429     0.991124  0.450000

Full Classification Report:
               precision    recall  f1-score   support

        No DR       1.00      0.98      0.99       172
         Mild       0.67      0.65      0.66        40
     Moderate       0.77      0.78      0.78       104
       Severe       0.41      0.77      0.54        22
Proliferative       0.75      0.32     